In [ ]:
#TODO: read parameters from practicemap.yaml into variables and use them
import pandas as pd
import numpy as np

df = pd.read_csv("../../maps/practicemap/practicemap.csv")
centerline_points = np.array((df.values + [-3.63, -8.52])/0.05, dtype=np.int32)

In [ ]:
VEHICLE_WIDTH = 0.27

In [ ]:
import cv2 as cv
practicemap = cv.imread("../../maps/practicemap/practicemap.pgm", cv.IMREAD_GRAYSCALE)

# for point in centerline_points:
#     practicemap[point[1], point[0]] = 1
# cv.imshow("practicemap", practicemap)
# cv.waitKey(0)
# cv.destroyAllWindows()

In [ ]:
from perception_data import *

def coords(t, a, b, c, d):
    t_floor = np.int32(t)
    t_frac = t - t_floor
    return np.array([ (a[I] + b[I]*f + c[I]*f**2 + d[I]*f**3) for i in range(t.shape[0]) if (I := (t_floor[i])%(a.shape[0]), f := t_frac[i]) ])

# def implemented_visualize_splines(points: np.ndarray, line_label: str, points_label: str, show_control_points: bool = False, dashed: bool = False):
#     Ainv = matAInv(points.shape[0])
#     centreline = Centreline(points.shape[0], points, None, None)
#     abcd_x = (Ainv @ q_comp(centreline, 0)).reshape((-1, 4))
#     abcd_y = (Ainv @ q_comp(centreline, 1)).reshape((-1, 4))

#     # Define the parametric points (x and y coordinates)
#     t_points = np.arange(points.shape[0])
#     x_points = np.array(points[:, 0])
#     y_points = np.array(points[:, 1])

#     # Generate values of t for plotting the spline
#     t_fine = np.linspace(t_points[0], t_points[-1] + 1, 1000)

#     # Evaluate the splines to get the points on the curve
#     x_fine = coords(t_fine, abcd_x[:, 0], abcd_x[:, 1], abcd_x[:, 2], abcd_x[:, 3])
#     y_fine = coords(t_fine, abcd_y[:, 0], abcd_y[:, 1], abcd_y[:, 2], abcd_y[:, 3])

#     # Plot the parametric spline
#     if dashed: plt.plot(x_fine, y_fine, '--', label=line_label)
#     else: plt.plot(x_fine, y_fine, label=line_label)
#     if show_control_points: plt.plot(x_points, y_points, 'o', label=points_label)
#     plt.xlabel('x')
#     plt.ylabel('y')
#     plt.legend()
#     plt.grid(True)
#     plt.axis('equal')  # Ensure the x and y axes have the same scale

def centerline_with_track_width(points: np.ndarray, map: np.ndarray, occ_thresh: int = 50, free_thresh: int = 2):
    Ainv = matAInv(points.shape[0])
    centreline = Centreline(points.shape[0], points, None, VEHICLE_WIDTH)
    q_x = q_comp(centreline, 0)
    q_y = q_comp(centreline, 1)
    x_d = first_derivatives(centreline, Ainv, q_x)
    y_d = first_derivatives(centreline, Ainv, q_y)
    centreline.calc_n(x_d, y_d)
    
    track_widths = []
    for i in range(centreline.N):
        x, y = points[i]
        n_x, n_y = centreline.n[i]

        t_step = 1/max(n_x, n_y)
        t = np.arange(0, max(map.shape[0], map.shape[1]))*t_step
        # TODO: check whether left_endpoint and right_endpoint are actually left and right
        normal_left = [x, y] + t*[n_x, n_y]
        normal_right = [x, y] - t*[n_x, n_y]
        for point in normal_left:
            if map[point[1], point[0]] > occ_thresh:
                left_endpoint = point
                break
        for point in normal_right:
            if map[point[1], point[0]] > occ_thresh:
                right_endpoint = point
                break
        
        track_width = np.linalg.norm(left_endpoint - right_endpoint) / 2
        track_widths.append(track_width)
    centreline.half_w_tr = track_widths
    return centreline